# Humanoid curriculum RL walking (MuJoCo MJX + Brax PPO)

Trains the learning-based walking controller from `rl/` in three curriculum phases:

1. **Posture and balance**: stand on both feet.
2. **Weight shifting and foot clearance**: step in place following the gait clock.
3. **Low-speed locomotion**: walk forward at 0.05-0.2 m/s.

All progress (checkpoints, `curriculum_state.json`, TensorBoard logs) is written to Google Drive, so
after a Colab disconnect re-run the setup cells and the training cell: it resumes from the last
completed chunk. See `rl/README.md` for the design and the tuning checklist.

**Runtime:** select *Runtime > Change runtime type > GPU* (T4 on the free tier).

In [ ]:
!nvidia-smi

## Configuration

In [ ]:
REPO_URL = 'https://github.com/Bjarne33/Bjarne33-3D-Bipedal-Walking.git'  # private repo: use a token URL
BRANCH = 'main'
RUN_NAME = 'walk_v1'
DRIVE_ROOT = '/content/drive/MyDrive/humanoid_rl'
NUM_ENVS = None  # None keeps ppo_config().num_envs; set from the benchmark below

## Setup (re-run after every disconnect)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
RUN_DIR = os.path.join(DRIVE_ROOT, RUN_NAME)
os.makedirs(RUN_DIR, exist_ok=True)
print('run dir:', RUN_DIR)

In [ ]:
%cd /content
![ -d repo ] || git clone --depth 1 -b {BRANCH} {REPO_URL} repo
!cd repo && git pull --ff-only
!pip install -q -e "repo/rl[cuda]"

In [ ]:
%env MUJOCO_GL=egl
%env XLA_PYTHON_CLIENT_PREALLOCATE=false
import jax
print(jax.__version__, jax.devices())
assert jax.devices()[0].platform == 'gpu', 'No GPU: check the runtime type and the jax[cuda12] install'

## Benchmark (once per GPU type)

Environment steps per second for several batch sizes (random actions, domain randomization on).
PPO throughput is lower (network updates), but the curve shows where the GPU saturates. Pick the
smallest `NUM_ENVS` near the plateau. Brax requires `batch_size * num_minibatches` (8192 with the
defaults) to be divisible by `num_envs`, so use a power of two up to 8192.

In [ ]:
from humanoid_rl import benchmark
results = benchmark.sweep((512, 1024, 2048, 4096, 8192), num_steps=100)

## Train

`curriculum.run` trains chunk by chunk (`curriculum_config().chunk_steps`), evaluates the
deterministic policy after each chunk and advances the phase when its thresholds are met.
Interrupting the cell is safe; re-running it resumes.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir {RUN_DIR}/tensorboard

In [ ]:
from humanoid_rl import curriculum

setup = curriculum.TrainingSetup()
if NUM_ENVS is not None:
    setup.ppo.num_envs = NUM_ENVS
state = curriculum.run(RUN_DIR, setup)
print(state.status, 'phase', state.phase, 'total steps', state.total_steps)

## Status

`budget_exceeded` means a phase did not reach its thresholds within `max_phase_steps`. Inspect the
metrics and videos, adjust rewards or thresholds (see the README tuning checklist), then continue
with `curriculum.run(RUN_DIR, setup, extend_budget=True)`.

In [ ]:
import json
with open(os.path.join(RUN_DIR, 'curriculum_state.json')) as f:
    saved = json.load(f)
print('status:', saved['status'], '| phase:', saved['phase'], '| steps:', saved['total_steps'])
for entry in saved['history'][-5:]:
    metrics = {k: round(v, 3) for k, v in entry['metrics'].items() if not k.startswith('reward/')}
    print(entry['chunk'], 'phase', entry['phase'], 'passed' if entry['passed'] else '', metrics)

## Evaluate, video and export of the latest checkpoint

In [ ]:
from humanoid_rl import train
train.main(['eval', '--run_dir', RUN_DIR])

In [ ]:
import mediapy
video = os.path.join(RUN_DIR, 'latest.mp4')
train.main(['video', '--run_dir', RUN_DIR, '--seconds', '10', '--command', '0.15', '--output', video])
mediapy.show_video(mediapy.read_video(video), fps=50)

In [ ]:
policy_path = os.path.join(RUN_DIR, 'policy.npz')
train.main(['export', '--run_dir', RUN_DIR, '--output', policy_path])

from humanoid_rl.export import NumpyPolicy
policy = NumpyPolicy(policy_path)
print({k: v for k, v in policy.meta.items() if k != 'joint_names'})